# Kelly vs. SBF vs. risk-neutral gamblers — a far-OTM call

Each round the player buys a one-period far-out-of-the-money call (or any binary lottery with the same payoff profile): with probability $p = 0.10$ the position pays $15\times$ the premium; with probability $0.90$ it expires worthless. Per dollar of premium, the gross return is
$$R = \begin{cases} 15 & \text{w.p. } 0.10 \\ 0 & \text{w.p. } 0.90 \end{cases}$$
The arithmetic per-dollar EV is $0.10 \cdot 15 - 1 = +0.50$ (a 50% per-round expected return — strongly positive, like a real alpha source). But the lose state is full wipeout, and it hits 90% of the time. Each player bets a fraction $f$ of wealth each round, so wealth evolves as
$$W_{t+1} \;=\; W_t \bigl(1 + f\,(R - 1)\bigr).$$
We simulate $N = 10{,}000$ players for each of three strategies:

- **Kelly** ($f^* \approx 0.0357$) — log-utility / $\gamma = 1$ optimum.
- **SBF** ($f^* \approx 0.0508$) — CRRA optimum at $\gamma = 0.75$ (about 42% above Kelly's bet).
- **Risk neutral** ($f = 1$) — maximizes arithmetic $E[W_T]$ by going all-in each round; one losing round wipes you out.

Note that the K=16 indifference benchmark from `Kelly_etc.ipynb` was for a *50-50* gamble; here the win probability is only 10%, so SBF is forced to be far more cautious than $f = 1$. Even so, betting more than Kelly on a 90%-loss-state lottery exposes the SBF agent to a meaningfully higher risk of ruin over 100 rounds. Below we play these strategies forward and see where the players end up.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(544)

In [2]:
initial_investment = 1000
T = 100         # rounds per player
N = 10_000      # players per strategy

# Lottery: P(R = 15) = 0.10, P(R = 0) = 0.90.  Per-dollar EV = +0.50.
returns = np.array([15.0, 0.0])
probs   = np.array([0.10, 0.90])

f_kelly = 0.0357    # log-utility optimum on this lottery
f_sbf   = 0.0508    # CRRA optimum at gamma = 0.75 on this lottery
f_rn    = 1.0       # risk-neutral: all-in each round

In [3]:
# Vectorized Monte Carlo: each row is one player's T draws of the underlying asset.
draws = rng.choice(returns, size=(N, T), p=probs)

kelly_multipliers = 1 + f_kelly * (draws - 1)
sbf_multipliers   = 1 + f_sbf   * (draws - 1)
rn_multipliers    = 1 + f_rn    * (draws - 1)   # equals draws itself when f=1

kelly_final = initial_investment * np.prod(kelly_multipliers, axis=1)
sbf_final   = initial_investment * np.prod(sbf_multipliers,   axis=1)
rn_final    = initial_investment * np.prod(rn_multipliers,    axis=1)

## Expected final wealth, survival, and "above start" rates

Returns are i.i.d., so $E[W_T] = W_0 \cdot (E[\text{multiplier}])^T$. The arithmetic expected per-round multiplier under fraction $f$ is
$$E[1 + f(R-1)] \;=\; 1 + f\bigl(E[R] - 1\bigr).$$
For this lottery $E[R] = 0.10 \cdot 15 + 0.90 \cdot 0 = 1.5$, so the per-round multiplier is $1 + 0.5\,f$ and $E[W_T] = 1000 \cdot (1 + 0.5\,f)^{100}$.

Two extra diagnostics matter here, since the lose state is full wipeout:

- **Survival rate** — fraction of players with $W_T > 0$.
- **Above-start rate** — fraction of players with $W_T > W_0$.

In [4]:
E_R = float(np.dot(probs, returns))   # arithmetic mean of R

def theoretical_E_WT(f):
    return initial_investment * (1 + f * (E_R - 1)) ** T

def diagnostics(final, f, label):
    return {
        "Strategy":            label,
        "Theoretical E[W_T]":  f"{theoretical_E_WT(f):.2e}",
        "Simulated E[W_T]":    f"{final.mean():.2e}",
        "Survival rate":       f"{(final > 0).mean():.4%}",
        "Above-start rate":    f"{(final > initial_investment).mean():.4%}",
    }

ev_table = pd.DataFrame([
    diagnostics(kelly_final, f_kelly, f"Kelly (f = {f_kelly})"),
    diagnostics(sbf_final,   f_sbf,   f"SBF (f = {f_sbf})"),
    diagnostics(rn_final,    f_rn,    f"Risk neutral (f = {f_rn})"),
])
print(ev_table.to_string(index=False))

              Strategy Theoretical E[W_T] Simulated E[W_T] Survival rate Above-start rate
    Kelly (f = 0.0357)           5.87e+03         5.88e+03     100.0000%         67.9200%
      SBF (f = 0.0508)           1.23e+04         1.19e+04     100.0000%         67.9200%
Risk neutral (f = 1.0)           4.07e+20         0.00e+00       0.0000%          0.0000%


## Percentiles of the wealth distribution

In [5]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9, 99.99]

def fmt(x):
    return f"{x:.2e}" if x > 0 else "0"

table = pd.DataFrame({
    "Percentile":            percentiles,
    "Kelly":                 [fmt(v) for v in np.percentile(kelly_final, percentiles)],
    "SBF":                   [fmt(v) for v in np.percentile(sbf_final,   percentiles)],
    "Risk neutral":          [fmt(v) for v in np.percentile(rn_final,    percentiles)],
})
print(table.to_string(index=False))

 Percentile    Kelly      SBF Risk neutral
       1.00 1.54e+02 5.75e+01            0
       5.00 2.40e+02 1.04e+02            0
      10.00 3.73e+02 1.87e+02            0
      25.00 9.03e+02 6.07e+02            0
      50.00 2.18e+03 1.97e+03            0
      75.00 5.29e+03 6.41e+03            0
      90.00 1.28e+04 2.08e+04            0
      95.00 1.99e+04 3.76e+04            0
      99.00 7.48e+04 2.20e+05            0
      99.90 1.81e+05 7.16e+05            0
      99.99 4.38e+05 2.33e+06            0


## Top 10 final wealth values among the 10,000 players (each strategy)

In [6]:
kelly_top10 = np.sort(kelly_final)[::-1][:10]
sbf_top10   = np.sort(sbf_final)[::-1][:10]
rn_top10    = np.sort(rn_final)[::-1][:10]

top10_table = pd.DataFrame({
    "Rank":         np.arange(1, 11),
    "Kelly":        [fmt(v) for v in kelly_top10],
    "SBF":          [fmt(v) for v in sbf_top10],
    "Risk neutral": [fmt(v) for v in rn_top10],
})
print(top10_table.to_string(index=False))

 Rank    Kelly      SBF Risk neutral
    1 4.38e+05 2.33e+06            0
    2 4.38e+05 2.33e+06            0
    3 2.81e+05 1.29e+06            0
    4 2.81e+05 1.29e+06            0
    5 2.81e+05 1.29e+06            0
    6 2.81e+05 1.29e+06            0
    7 2.81e+05 1.29e+06            0
    8 2.81e+05 1.29e+06            0
    9 2.81e+05 1.29e+06            0
   10 1.81e+05 7.16e+05            0


**Takeaway.** This far-OTM-call lottery has a strongly positive arithmetic per-round EV (+50%), the lose state is full wipeout, and it hits 90% of the time:

- **Risk neutral** ($f = 1$) maximizes theoretical $E[W_T]$ at $\sim 4 \times 10^{20}$, but the only way to realize it is to win all 100 rounds — probability $0.1^{100} = 10^{-100}$. With $N = 10{,}000$ players, *every single one* is wiped out.
- **Kelly** ($f \approx 0.036$) and **SBF** ($f \approx 0.051$) both mathematically survive every sample path (since fraction-of-wealth betting with $f < 1$ never literally hits zero), and have identical *above-start* rates of ~68%. But the percentile distributions tell the real story: SBF's worst-case losses are systematically deeper than Kelly's. The 1st-percentile Kelly player ends with \$154; the 1st-percentile SBF player ends with \$58 — a 62% bigger drawdown for the same probability event. The 5th and 10th percentiles show the same pattern.
- The price SBF pays for that worse left tail is a heavier *right* tail: above the 75th percentile she dominates Kelly, and at the very top (top 10 of 10,000) SBF's wealth runs roughly 5× Kelly's.

The lesson: even when "ruin" in the literal $W_T = 0$ sense is impossible, betting more aggressively than Kelly on a positive-EV lottery with frequent loss states produces *much* deeper bottom-quartile outcomes — what financial industry observers would call near-ruin or capital impairment. The K=16 indifference benchmark doesn't bind here because the win probability isn't 50%; it's the persistently negative compounding effect of frequent losses that punishes the SBF agent's larger fraction.